# Sizing a myLedger node

**What this answers:** how much memory and how much disk, for a stated demand and a stated policy.

**What it does not answer:** throughput and tail latency. Those are not arithmetic — design notes
§10 records a hardware profile that was built, measured and *refused*, because the stages' cache
misses overlap and the formula priced each at full latency. Ask `ledgersim` instead, which runs the
real reactor on a virtual clock.

## The split

| | owns | why |
|---|---|---|
| the code | what one unit costs | it is `size_of`, known at build time |
| this notebook | how many units | it follows from a rate, a lifetime and a retention |

Nothing here hard-codes a byte count. They come from `sizing/units.json`, which is
`ledgerfio layout --json` with the commit it was taken at. Refresh it with `make sizing-units`
after changing a sized struct; `make verify` fails if it is stale.


In [ ]:
# Find `model.py` whichever directory the kernel started in: VS Code opens a notebook at its own
# folder by default, but `jupyter.notebookFileRoot` can be set to the workspace root and then a bare
# '.' is the wrong place. Searching is one line and removes the setting from the picture.
import os, sys
here = os.getcwd()
while not os.path.exists(os.path.join(here, 'model.py')):
    parent = os.path.dirname(here)
    if parent == here:
        raise SystemExit('open this notebook from the repository, beside sizing/model.py')
    here = os.path.join(here, 'sizing') if os.path.isdir(os.path.join(here, 'sizing')) else parent
sys.path.insert(0, here)

from model import (Lifetimes, Demand, Policy, Dials, Sizing, report,
                   residency_curve, print_residency_curve,
                   buckets_for, index_slots_for, gigabytes, check_bucket_rule, load_units)

units = load_units()
check_bucket_rule(units)   # the one piece of the code's arithmetic reproduced here
print(units['source'], 'at', units['commit'][:12])
print(len(units['parts']), 'sized structures')


## The inputs

**Three rates, not one.** A peak decides what is held in flight, the busiest hour decides the
windows an hour wide, and the day decides retention. A deployment whose peak is eighty-six times
its mean is sized eighty-six times wrong by whichever single number it picks.

**Dials are outputs, not inputs.** `Dials` is here so a plan can be checked against the ceilings a
node is configured with — a dial below what demand requires is where the node refuses work.


In [ ]:
# How long holds live, as a CURVE. Points are (hours, share resolved by then).
# This one input replaced two guesses -- 'how long does a hold live' and 'what share survives' are
# the same fact read at two points, and asking separately let them disagree.
lifetimes = Lifetimes([
    (1,      0.50),   # half are voided or settled within the hour
    (4,      0.70),
    (24,     0.88),
    (24 * 7, 0.96),   # the rest never resolve -- which is what retention is for
])

demand = Demand(
    # --- the three rates: one number each, and they are not the same number ---
    peak_rate=300_000,           # tx/s at the peak
    peak_seconds=60,             # how long that peak holds. rate alone decides nothing
    busiest_hour_tx=30_000_000,  # tx in the busiest hour -- the hour-wide windows follow this
    daily_tx=300_000_000,        # tx in a day -- retention and disk follow this
    accounts=10_000_000,         # the working set: accounts, not traffic

    # --- the shape of that traffic ---
    lifetimes=lifetimes,
    hold_share=1.0,              # of every tx, the share that CREATES a hold.
                                 #   a plain transfer creates none; so does a settle.
    records_per_hold=1.0,        # records one hold appends over its life:
                                 #   1, plus 1 per PARTIAL settle. resolving in full appends none.
    commit_latency_seconds=0.010,# submit to commit. x peak rate = requests in flight
)

policy = Policy(
    retention_days=30,           # THE FIRST DECISION: how long a hold may live before expiry voids it
    grace_days=1,                #   slack before the record is deleted, so expiry is never early
    flush_window_hours=1,        # a RECOVERY bound: what a restart replays
    residency_hours=24,          # a LATENCY bound -- and see the residency section below: this one
                                 #   is an ANSWER, not a preference
    idem_window_hours=1,         # duplicate detection -- the code does not enforce this yet
    snapshot_every_effects=1_000_000,
)

print(demand.sanity() or 'inputs are consistent')
for hours in (1, 4, 24, 24 * 30):
    print(f'{hours:>5}h  {lifetimes.resolved_by(hours):.0%} resolved')


In [ ]:
sizing = Sizing(demand, policy, Dials())
print(report(sizing))


## Changing a number

Any unit cost can be overridden to ask *what if this struct were smaller*. **The report says so.**
A hypothetical printed beside measurements reads as a measurement, which is the same rule every
benchmark here follows when it prints its thread placement.


In [ ]:
what_if = Sizing(demand, policy, Dials(), overrides={'pending index': 4})
print(report(what_if).splitlines()[0])
print()
print(f"measured  {gigabytes(sizing.memory_bytes):.2f} GB")
print(f"overridden {gigabytes(what_if.memory_bytes):.2f} GB")


## The staircase

A hash table rounds its bucket count to a power of two, so **one percent more entries can double
the memory**. The cuckoo index steps too, four times more coarsely. A single point tells you
nothing about which side of a step it is on — sweep.


In [ ]:
print(f"{'daily tx':>14}{'live holds':>16}{'index slots':>16}{'index GB':>11}")
for daily in (100_000_000, 200_000_000, 300_000_000, 400_000_000, 600_000_000, 800_000_000):
    one = Sizing(Demand(300_000, 60, min(daily, 30_000_000), daily, 10_000_000, lifetimes), policy)
    line = dict(one.lines_by_name)['pending index']
    print(f'{daily:>14,}{one.live_holds:>16,}{line.count:>16,}{gigabytes(line.bytes):>11.2f}')


### The idem map, and why the peak's *duration* is an input

The idem window is an hour, so its count is the busiest hour's transactions — there is no queue to
absorb a peak into. This is the structure where the peak-to-mean ratio decides everything.


In [ ]:
print(f"{'busiest hour tx':>18}{'buckets':>16}{'idem GB':>10}")
for hour in (3_000_000, 10_000_000, 30_000_000, 100_000_000, 300_000_000, 1_080_000_000):
    one = Sizing(Demand(300_000, 60, hour, max(hour, 300_000_000), 10_000_000, lifetimes), policy)
    line = dict(one.lines_by_name)['idem keys']
    print(f'{hour:>18,}{line.count:>16,}{gigabytes(line.bytes):>10.2f}')
print()
print('a peak of 300k/s sustained for a whole hour is the last row: 1.08G keys.')


## What is not sized here

- **Throughput and the tail** — `ledgersim capacity` and `ledgersim require`, which run the real
  reactor rather than a formula.
- **`skew`** — hot-account concentration costs lane contention, which is latency, not bytes.
- **The idem window** — one hour is the intended window and the count above assumes it, but the
  rotating generations that would enforce it are not built. Today the map only grows.
- **`kept log`** — there is no compaction, so its count is a snapshot cadence rather than a steady
  state.


## Residency is an answer, not a preference

The lifetime curve is what makes this computable. A resolution inside the residency window is
answered from memory; one after it **reads the device**. So the device read rate is a subtraction
on the curve — resolutions that land in the gap between residency ending and expiry taking the
hold — and widening residency buys reads at a price in blocks.

It is printed as a curve rather than solved. What a read is worth depends on the device and on the
tail somebody is holding, and neither is in this file — the same refusal design notes §10 makes one
level down.


In [ ]:
print_residency_curve(residency_curve(demand, policy, [1, 2, 4, 8, 24, 72, 168]))


### Retention moves the other end

Retention decides how long a hold may live before expiry voids it — **the first decision**, because
everything on disk is multiplied by it. Thirty days against ten is three times the segment files,
and it also moves how many holds are live at once, which is the index that cannot grow.


In [ ]:
print(f"{'retention':>10}{'expiry voids':>14}{'live holds':>16}{'index GB':>10}{'disk GB':>10}")
for days in (7, 10, 30, 90, 180):
    moved = Policy(retention_days=days, grace_days=1)
    one = Sizing(demand, moved)
    index = dict(one.lines_by_name)['pending index']
    print(f'{days:>9}d{one.survivor_share:>14.1%}{one.live_holds:>16,}'
          f'{gigabytes(index.bytes):>10.2f}{gigabytes(one.disk_bytes):>10.1f}')
